# LoRA Compress - Q4 Quantized Model Adaptive Autoresearch

This version loads a Q4 quantized model and trains LoRA to match the **quantized weights**.

**Hypothesis:** Since Q4 weights already have quantization error, they may be more
compressible with low-rank approximation (lower intrinsic rank).

**Features:**
- Uses bitsandbytes for Q4 quantization
- Same OAT (Ordered Adaptive Testing) strategy
- Compare Q4 vs FP16 compression difficulty

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

print(f'✅ Drive mounted: {DRIVE_BASE}')

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git 2>/dev/null || true
%cd loracompress

# Install dependencies including bitsandbytes for quantization
!pip install -q transformers torch optuna tqdm bitsandbytes accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import random
import numpy as np
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print(f'🔥 PyTorch: {torch.__version__}')
print(f'🔥 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Step 4: Load Q4 Quantized Model

We use bitsandbytes to load the model in 4-bit precision.
The dequantized weights are what we'll try to compress with LoRA.

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

# Q4 quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Nested quantization
    bnb_4bit_quant_type='nf4',  # NF4 is optimal for weights
)

print('Loading Q4 quantized model...')
print('(This downloads quantized weights and decompresses them to FP16 for computation)')

model_q4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

print(f'\n✅ Model loaded')
print(f'   Device map: {model_q4.hf_device_map if hasattr(model_q4, "hf_device_map") else "auto"}')

## Step 5: Also Load FP16 Model for Comparison

In [ ]:
# Load same model in FP16 for comparison
print('Loading FP16 model for comparison...')

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',  # Keep on CPU to save GPU memory
    trust_remote_code=True,
)

print('✅ FP16 model loaded (on CPU)')

## Step 6: Layer Browser with Q4 vs FP16 Comparison

In [ ]:
def get_layer_singular_values(weight, max_rank=128):
    """Compute singular values for compressibility analysis."""
    w = weight.float()
    try:
        U, S, Vh = torch.linalg.svd(w, full_matrices=False)
        # Variance explained by different ranks
        total = (S ** 2).sum()
        var_64 = (S[:64] ** 2).sum() / total * 100
        var_128 = (S[:128] ** 2).sum() / total * 100 if len(S) >= 128 else 100.0
        # Estimate L1 error for rank-64 approximation
        S_64 = torch.zeros_like(S)
        S_64[:64] = S[:64]
        W_64 = U @ torch.diag(S_64) @ Vh
        l1_err_64 = torch.mean(torch.abs(W_64 - w)).item() / torch.mean(torch.abs(w)).item() * 100
        return var_64.item(), var_128.item(), l1_err_64
    except:
        return 0, 0, 100

print('Analyzing layers: Q4 vs FP16 compressibility\n')
print('=' * 100)
print(f"{'Layer':<45} {'Q4 R64%':>8} {'FP16 R64%':>10} {'Q4 L1':>8} {'FP16 L1':>10} {'Q4 Easier?':>12}")
print('=' * 100)

layer_comparisons = []

for (name_q4, param_q4), (name_fp16, param_fp16) in zip(
    model_q4.named_parameters(), model_fp16.named_parameters()
):
    if len(param_q4.shape) != 2 or param_q4.shape[0] < 100:
        continue
    
    # Get weights (Q4 is already dequantized when accessed)
    w_q4 = param_q4.data.cpu().float()
    w_fp16 = param_fp16.data.cpu().float()
    
    var64_q4, var128_q4, err64_q4 = get_layer_singular_values(w_q4)
    var64_fp16, var128_fp16, err64_fp16 = get_layer_singular_values(w_fp16)
    
    # Q4 is easier if rank-64 captures more variance OR has lower L1 error
    q4_easier = var64_q4 > var64_fp16 or err64_q4 < err64_fp16
    
    layer_comparisons.append({
        'name': name_q4,
        'shape': tuple(param_q4.shape),
        'q4_var64': var64_q4,
        'fp16_var64': var64_fp16,
        'q4_err64': err64_q4,
        'fp16_err64': err64_fp16,
        'q4_easier': q4_easier,
        'w_q4': w_q4,
        'w_fp16': w_fp16,
    })
    
    # Print first 20 layers
    if len(layer_comparisons) <= 20:
        easier_mark = '✓' if q4_easier else '✗'
        name_short = name_q4.split('.')[-2] + '.' + name_q4.split('.')[-1] if '.' in name_q4 else name_q4[:40]
        print(f"{name_short:<45} {var64_q4:>8.1f} {var64_fp16:>10.1f} {err64_q4:>8.1f} {err64_fp16:>10.1f} {easier_mark:>12}")

print('\n' + '=' * 100)

# Summary
q4_easier_count = sum(1 for l in layer_comparisons if l['q4_easier'])
print(f"\n📊 Summary: Q4 is easier to compress for {q4_easier_count}/{len(layer_comparisons)} layers ({q4_easier_count/len(layer_comparisons)*100:.1f}%)")

# Find best Q4 layer (high variance captured, lower error than FP16)
good_q4_layers = [l for l in layer_comparisons if l['q4_var64'] > 90 and l['q4_easier']]
if good_q4_layers:
    best_q4 = max(good_q4_layers, key=lambda x: x['q4_var64'] - x['fp16_var64'])
    print(f"\n🏆 Best Q4 compression candidate: {best_q4['name']}")
    print(f"   Q4 var: {best_q4['q4_var64']:.1f}%, FP16 var: {best_q4['fp16_var64']:.1f}%")
    print(f"   Q4 L1: {best_q4['q4_err64']:.1f}%, FP16 L1: {best_q4['fp16_err64']:.1f}%")

## Step 7: Select Target Layer

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LAYER SELECTION
# ═══════════════════════════════════════════════════════════════

# Option 1: Select by index from layer_comparisons list
LAYER_INDEX = None

# Option 2: Select by name pattern
LAYER_PATTERN = 'layers.15'  # Try 'layers.20', 'q_proj', etc.

# Option 3: Auto-select best Q4 layer
AUTO_SELECT_BEST_Q4 = True

# ═══════════════════════════════════════════════════════════════

def select_layer(comparisons, index=None, pattern=None, auto_best=False):
    if auto_best:
        # Find Q4 layer with biggest advantage over FP16
        good_layers = [l for l in comparisons if l['q4_var64'] > 85]
        if good_layers:
            return max(good_layers, key=lambda x: x['q4_var64'] - x['fp16_var64'])
    
    if index is not None and index < len(comparisons):
        return comparisons[index]
    
    if pattern:
        matches = [l for l in comparisons if pattern in l['name']]
        if matches:
            # Prefer Q4-easier layers
            easier = [m for m in matches if m['q4_easier']]
            if easier:
                return max(easier, key=lambda x: x['q4_var64'])
            return matches[0]
    
    # Default: pick a medium layer where Q4 has advantage
    easier_mid = [l for l in comparisons if l['q4_easier'] and 'layers.10' in l['name'] or 'layers.15' in l['name']]
    if easier_mid:
        return easier_mid[len(easier_mid)//2]
    return comparisons[len(comparisons)//2]

selected = select_layer(layer_comparisons, LAYER_INDEX, LAYER_PATTERN, AUTO_SELECT_BEST_Q4)

print(f"✅ Selected layer: {selected['name']}")
print(f"   Shape: {selected['shape']}")
print(f"\n📊 Q4 vs FP16 Comparison:")
print(f"   Q4 rank-64 variance:  {selected['q4_var64']:.1f}%")
print(f"   FP16 rank-64 variance: {selected['fp16_var64']:.1f}%")
print(f"   Q4 rank-64 L1 error:  {selected['q4_err64']:.1f}%")
print(f"   FP16 rank-64 L1 error: {selected['fp16_err64']:.1f}%")
print(f"   Q4 is easier: {selected['q4_easier']}\n")

# Use Q4 weights for training
target_weight_q4 = selected['w_q4']
target_weight_fp16 = selected['w_fp16']

print(f"Target weight (Q4): {target_weight_q4.shape}")

## Step 8: Adaptive Explorer & Training Functions

In [ ]:
class AdaptiveExplorer:
    """Intelligent hyperparameter explorer with OAT."""

    def __init__(self, lr_min=0.0005, lr_max=0.8, rank_min=16, rank_max=512, aggressive_ratio=0.3):
        self.lr_min = lr_min
        self.lr_max = lr_max
        self.rank_min = rank_min
        self.rank_max = rank_max
        self.aggressive_ratio = aggressive_ratio
        self.lr_history = []
        self.rank_history = []
        self.lr_upper_boundary = None
        self.lr_optimal_region = None
        self.rank_effectiveness = {}
        self.oat_phase = 'explore_lr_up'
        self.oat_test_values = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64]
        self.oat_test_idx = 0
        self.oat_best_lr = None

    def sample_lr_two_region(self, trial):
        if random.random() < self.aggressive_ratio:
            return trial.suggest_float('lr', 0.1, self.lr_max, log=True)
        return trial.suggest_float('lr', self.lr_min, 0.1, log=True)

    def sample_lr_adaptive(self, trial):
        if self.lr_optimal_region and random.random() < 0.6:
            low, high = self.lr_optimal_region
            return trial.suggest_float('lr', max(self.lr_min, low * 0.8),
                                        min(self.lr_max, high * 1.2), log=True)
        return self.sample_lr_two_region(trial)

    def sample_rank_adaptive(self, trial):
        if self.rank_effectiveness and random.random() < 0.5:
            sorted_ranks = sorted(self.rank_effectiveness.items(), key=lambda x: x[1])
            good_ranks = [r for r, err in sorted_ranks[:3] if err < 50]
            if good_ranks:
                base_rank = random.choice(good_ranks)
                low = max(self.rank_min, int(base_rank * 0.75))
                high = min(self.rank_max, int(base_rank * 1.25))
                return trial.suggest_int('rank', low, high, log=True)
        return trial.suggest_int('rank', self.rank_min, self.rank_max, log=True)

    def update_from_result(self, config, error):
        lr = config.get('lr')
        rank = config.get('rank')
        if lr:
            self.lr_history.append((lr, error))
        if rank:
            self.rank_history.append((rank, error))
            self.rank_effectiveness[rank] = error
        if len(self.lr_history) >= 5:
            sorted_lrs = sorted(self.lr_history, key=lambda x: x[0])
            best_err = min(e for _, e in sorted_lrs)
            good_lrs = [lr for lr, err in sorted_lrs if err < best_err * 1.5]
            if good_lrs:
                self.lr_optimal_region = (min(good_lrs), max(good_lrs))

    def get_oat_suggestion(self, trial_number):
        if trial_number == 0:
            return {'rank': 64, 'lr': 0.01, 'epochs': 500}

        if self.oat_phase == 'explore_lr_up':
            if len(self.lr_history) >= 2:
                if self.lr_history[-1][1] > self.lr_history[-2][1] * 1.3:
                    self.lr_upper_boundary = self.lr_history[-1][0]
                    self.oat_best_lr = self.lr_history[-2][0]
                    self.oat_phase = 'refine_lr_down'
                    self.oat_test_values = [self.oat_best_lr * (0.95 ** i) for i in range(1, 10)]
                    self.oat_test_idx = 0
                    print(f'  [OAT] Upper boundary at {self.lr_upper_boundary:.4f}')
            if self.oat_test_idx < len(self.oat_test_values):
                lr = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': 64, 'lr': lr, 'epochs': 500}

        if self.oat_phase == 'refine_lr_down':
            if len(self.lr_history) >= 2 and self.lr_history[-1][1] > self.lr_history[-2][1]:
                self.oat_best_lr = self.lr_history[-2][0]
                self.oat_phase = 'explore_rank'
                self.oat_test_values = [16, 24, 32, 48, 64, 96, 128, 192, 256]
                self.oat_test_idx = 0
                print(f'  [OAT] Best LR: {self.oat_best_lr:.4f}, exploring ranks')
            if self.oat_test_idx < len(self.oat_test_values):
                lr = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': 64, 'lr': lr, 'epochs': 500}

        if self.oat_phase == 'explore_rank':
            if self.oat_test_idx < len(self.oat_test_values):
                rank = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': rank, 'lr': self.oat_best_lr or 0.01, 'epochs': 500}
            self.oat_phase = 'exploit'
            print('  [OAT] Switching to exploitation')
        return None


def compute_l1_error(W_approx, target):
    l1_error = torch.mean(torch.abs(W_approx - target)).item()
    mean_abs_target = torch.mean(torch.abs(target)).item()
    return (l1_error / mean_abs_target * 100) if mean_abs_target > 0 else float('inf')


def train_lora(target_weight, rank, lr, epochs, device='cuda', scheduler_type=None):
    d, k = target_weight.shape
    target = target_weight.float().to(device)
    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)
    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)
    optimizer = torch.optim.AdamW([A, B], lr=lr)
    
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)
    elif scheduler_type == 'linear':
        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=epochs)
    
    best_loss = float('inf')
    best_A, best_B = None, None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        W_approx = torch.matmul(B, A)
        loss = F.mse_loss(W_approx, target)
        
        if not torch.isfinite(loss):
            return float('inf'), 0
        
        loss.backward()
        optimizer.step()
        
        if scheduler:
            scheduler.step()
        
        current = loss.item()
        if current < best_loss - 1e-9:
            best_loss = current
            best_A = A.detach().clone()
            best_B = B.detach().clone()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epoch >= 200 and epochs_no_improve >= 100:
            break
    
    if best_A is None:
        return float('inf'), 0
    
    with torch.no_grad():
        W_best = torch.matmul(best_B, best_A)
        l1_error = compute_l1_error(W_best, target)
    
    return l1_error, epoch + 1

print('✅ Functions loaded')

## Step 9: Quick Test - Compare Q4 vs FP16 Training

In [ ]:
# Quick sanity check: train rank-64 on both Q4 and FP16 weights
print('Quick comparison: Rank-64 on Q4 vs FP16 weights\n')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

for test_rank in [32, 64]:
    print(f'\nRank {test_rank}:')
    
    # Train on Q4
    err_q4, epochs_q4 = train_lora(target_weight_q4, test_rank, 0.01, 500, device)
    print(f'  Q4:   {err_q4:.2f}% L1 error ({epochs_q4} epochs)')
    
    # Train on FP16
    err_fp16, epochs_fp16 = train_lora(target_weight_fp16, test_rank, 0.01, 500, device)
    print(f'  FP16: {err_fp16:.2f}% L1 error ({epochs_fp16} epochs)')
    
    if err_q4 < err_fp16:
        print(f'  ✅ Q4 is better by {err_fp16 - err_q4:.2f}%')
    else:
        print(f'  ❌ FP16 is better by {err_q4 - err_fp16:.2f}%')

## Step 10: Configuration & Run Autoresearch

In [ ]:
# Configuration
TARGET_QUALITY = 5.0
N_TRIALS = 50
EXPLORATION_MODE = 'oat'  # 'oat', 'adaptive', 'two_region'

# Adjust rank range based on layer difficulty
if selected['q4_var64'] > 90:
    RANK_MIN, RANK_MAX = 16, 128  # Easy layer
elif selected['q4_var64'] > 80:
    RANK_MIN, RANK_MAX = 32, 256  # Medium layer
else:
    RANK_MIN, RANK_MAX = 64, 512  # Hard layer

print(f'Rank range: {RANK_MIN} - {RANK_MAX} (based on Q4 variance: {selected["q4_var64"]:.1f}%)')

explorer = AdaptiveExplorer(lr_min=0.0005, lr_max=0.8,
                            rank_min=RANK_MIN, rank_max=RANK_MAX)

# Use Q4 weights for optimization
target_weight_gpu = target_weight_q4.to(device)

def objective(trial):
    trial_num = len(explorer.lr_history)
    
    if EXPLORATION_MODE == 'oat':
        oat_config = explorer.get_oat_suggestion(trial_num)
        if oat_config:
            rank = oat_config['rank']
            lr = oat_config['lr']
            epochs = oat_config['epochs']
            scheduler = None
            print(f'  [OAT {explorer.oat_phase}] r={rank}, lr={lr:.4f}')
        else:
            rank = explorer.sample_rank_adaptive(trial)
            lr = explorer.sample_lr_adaptive(trial)
            epochs = trial.suggest_int('epochs', 200, 2000, log=True)
            scheduler = trial.suggest_categorical('scheduler', [None, 'cosine', 'linear'])
    elif EXPLORATION_MODE == 'two_region':
        rank = trial.suggest_int('rank', RANK_MIN, RANK_MAX, log=True)
        lr = explorer.sample_lr_two_region(trial)
        epochs = trial.suggest_int('epochs', 200, 2000, log=True)
        scheduler = None
    else:
        rank = trial.suggest_int('rank', RANK_MIN, RANK_MAX, log=True)
        lr = trial.suggest_float('lr', 0.0005, 0.8, log=True)
        epochs = trial.suggest_int('epochs', 200, 2000, log=True)
        scheduler = None
    
    error, actual_epochs = train_lora(target_weight_gpu, rank, lr, epochs, device, scheduler)
    
    config = {'rank': rank, 'lr': lr, 'epochs': epochs}
    explorer.update_from_result(config, error)
    
    d, k = target_weight_q4.shape
    compression = (d * k) / (rank * (d + k))
    
    if error <= TARGET_QUALITY:
        score = -compression * 10
    else:
        score = 1000 + (error - TARGET_QUALITY) * 100
    
    trial.set_user_attr('error', error)
    trial.set_user_attr('compression', compression)
    trial.set_user_attr('actual_epochs', actual_epochs)
    return score

print('✅ Objective ready')

In [ ]:
# Run optimization
import os

study_name = f'q4_{EXPLORATION_MODE}_{TARGET_QUALITY}pct_{LAYER_PATTERN.replace(".", "_")}'
db_file = f'{DRIVE_BASE}/databases/{study_name}.db'

os.makedirs(os.path.dirname(db_file), exist_ok=True)

study = optuna.create_study(
    study_name=study_name,
    storage=f'sqlite:///{db_file}',
    direction='minimize',
    load_if_exists=True
)

print(f'\n🚀 Running {N_TRIALS} trials on Q4 weights')
print(f'   Target: <{TARGET_QUALITY}% L1 error')
print(f'   Layer: {selected["name"]}')
print(f'   Q4 vs FP16 advantage: {selected["q4_var64"] - selected["fp16_var64"]:.1f}% variance\n')

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best result: {study.best_trial.user_attrs["error"]:.2f}% L1 error')
print(f'   Config: rank={study.best_params.get("rank")}, lr={study.best_params.get("lr"):.4f}')
print(f'   Compression: {study.best_trial.user_attrs["compression"]:.1f}x')

## Step 11: Results Summary & Comparison

In [ ]:
# Collect results
results = []
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        results.append({
            'rank': trial.params.get('rank'),
            'lr': trial.params.get('lr'),
            'error': trial.user_attrs.get('error'),
            'compression': trial.user_attrs.get('compression'),
            'score': trial.value,
        })

results.sort(key=lambda x: x['error'])

print('\n' + '=' * 80)
print(f'🏆 TOP 10 RESULTS - Q4 Quantized Model')
print('=' * 80)
print(f"{'Rank':>6} {'LR':>10} {'Error%':>10} {'Compr':>10} {'Score':>10}")
print('-' * 80)

for r in results[:10]:
    m = '✓' if r['error'] <= TARGET_QUALITY else '✗'
    print(f"{r['rank']:>6} {r['lr']:>10.4f} {r['error']:>9.2f}{m} {r['compression']:>9.1f}x {r['score']:>10.1f}")

# Compare with theoretical FP16 performance
print('\n' + '=' * 80)
print('📊 Q4 vs FP16 COMPARISON')
print('=' * 80)
print(f"Layer: {selected['name']}")
print(f"Shape: {selected['shape']}")
print()
print(f"Rank-64 SVD variance:")
print(f"  Q4:   {selected['q4_var64']:.1f}%")
print(f"  FP16: {selected['fp16_var64']:.1f}%")
print(f"  Difference: {selected['q4_var64'] - selected['fp16_var64']:+.1f}%")
print()
print(f"Rank-64 LoRA L1 error (estimated):")
print(f"  Q4:   {selected['q4_err64']:.1f}%")
print(f"  FP16: {selected['fp16_err64']:.1f}%")
print(f"  Difference: {selected['fp16_err64'] - selected['q4_err64']:+.1f}%")

if study.best_trial.user_attrs['error'] < selected['fp16_err64']:
    print(f"\n✅ SUCCESS: Best LoRA error ({study.best_trial.user_attrs['error']:.1f}%) beats FP16 rank-64 ({selected['fp16_err64']:.1f}%)")
else:
    print(f"\n⚠️  LoRA error ({study.best_trial.user_attrs['error']:.1f}%) still higher than FP16 rank-64 ({selected['fp16_err64']:.1f}%)")

In [ ]:
# Save results
output = {
    'model': MODEL_NAME,
    'quantization': 'Q4_NF4',
    'layer_name': selected['name'],
    'layer_shape': selected['shape'],
    'target_quality': TARGET_QUALITY,
    'exploration_mode': EXPLORATION_MODE,
    'q4_vs_fp16': {
        'q4_var64': selected['q4_var64'],
        'fp16_var64': selected['fp16_var64'],
        'q4_err64': selected['q4_err64'],
        'fp16_err64': selected['fp16_err64'],
        'q4_easier': selected['q4_easier'],
    },
    'best_config': {
        'rank': study.best_params.get('rank'),
        'lr': study.best_params.get('lr'),
        'error': study.best_trial.user_attrs.get('error'),
        'compression': study.best_trial.user_attrs.get('compression'),
    },
    'all_results': results,
}

output_file = f'{DRIVE_BASE}/results/q4_{study_name}.json'
with open(output_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f'✅ Results saved: {output_file}')